# Depth-First Search and Components


```{contents}
:local:
:depth: 2
```


```{index} Finish One Branch Before Returning to Its Alternatives
```



```{index} Watch the Suspended Neighbor Loop
```

## Watch the Suspended Neighbor Loop


Depth-first search is ordinary recursive control flow applied to a graph. A call must remember which neighbor it is considering while a child explores its branch. This connects the suspended calls in Chapter 17 to Chapter 20's cycles and shared destinations. A discovered set avoids repeated discovery; it does not erase the caller's unfinished neighbor loop.

Use the undirected diamond A-B, A-C, B-D, C-D, and isolated E. In each array below, the listed neighbor order determines the trace. Watch enter and exit events separately. A vertex is entered once; returning from a child resumes an already active parent rather than entering that parent again.


In [1]:
using System;
using System.Collections.Generic;
var neighbors = new Dictionary<string, string[]>
{
    ["A"] = new[] { "B", "C" }, ["B"] = new[] { "A", "D" },
    ["C"] = new[] { "A", "D" }, ["D"] = new[] { "B", "C" }, ["E"] = Array.Empty<string>()
};
var seen = new HashSet<string>();
void Visit(string room, int depth)
{
    seen.Add(room);
    Console.WriteLine($"Enter {room} depth {depth}");
    foreach (string next in neighbors[room])
        if (!seen.Contains(next)) Visit(next, depth + 1);
    Console.WriteLine($"Exit {room}");
}
Visit("A", 0);
Console.WriteLine($"E reached: {seen.Contains("E")}");


Enter A depth 0
Enter B depth 1
Enter D depth 2
Enter C depth 3
Exit C
Exit D
Exit B
Exit A
E reached: False


C is first entered from D, not from A. A's neighbor loop has not finished, but C is already discovered by the time A resumes and considers it. C's DFS depth is three although A has a direct edge to C. A first DFS route therefore need not minimize hops.

| Event | Active frames from oldest to newest | What waits in the caller? |
| --- | --- | --- |
| Enter A | A | A will consider C after B returns |
| Enter B | A,B | B will finish after D returns |
| Enter D | A,B,D | D will finish after C returns |
| Enter C | A,B,D,C | C finds no undiscovered neighbor |
| Exit C | A,B,D | D resumes its neighbor loop |
| Exit D | A,B | B resumes |
| Exit B | A | A sees that C is already discovered |
| Exit A | Empty | Source traversal is complete |

Do not use the number of discovered vertices as the current stack depth. Four vertices have been discovered after C exits, but only three calls remain active at that point. Likewise, a visited neighbor alone is not enough to classify a directed cycle: completed and active destinations have different meanings.


```{index} Extend Reachability into a Component Report
```

## Extend Reachability into a Component Report


For an undirected graph, a component is the maximal group reachable from any of its members. Keep the discovered set across searches, then start another search from each undiscovered vertex. Starting a new traversal from a vertex that is already discovered would split or duplicate a component.

Guided task: after searching from A, which source starts another component? What is the answer for an empty graph? Then predict the report after adding isolated F. The following standalone example makes the labels visible without the full graph class.


In [1]:
using System;
using System.Collections.Generic;
var neighbors = new Dictionary<string, string[]>
{
    ["A"] = new[] { "B", "C" }, ["B"] = new[] { "A", "D" },
    ["C"] = new[] { "A", "D" }, ["D"] = new[] { "B", "C" }, ["E"] = Array.Empty<string>()
};
var component = new Dictionary<string, int>();
void Label(string room, int id)
{
    component[room] = id;
    foreach (string next in neighbors[room])
        if (!component.ContainsKey(next)) Label(next, id);
}
int count = 0;
foreach (string room in new[] { "A", "B", "C", "D", "E" })
    if (!component.ContainsKey(room)) Label(room, ++count);
foreach (string room in new[] { "A", "B", "C", "D", "E" })
    Console.WriteLine($"{room}: component {component[room]}");
Console.WriteLine($"Components: {count}");


A: component 1
B: component 1
C: component 1
D: component 1
E: component 2
Components: 2


E starts component two. An empty graph starts none; adding isolated F produces three. Component numbers depend on the declared start order, but membership equivalence does not: A and D must share a component and E must remain separate. Before independent practice, explain why these symmetric edges support undirected components while an arbitrary directed reachability report does not compute strongly connected components. The full implementations below retain their stronger input validation and source/path exercises.


```{index} Finish One Branch Before Returning to Its Alternatives
```

## Finish One Branch Before Returning to Its Alternatives



Depth-first search follows an unvisited neighbor, completes the exploration reachable through that choice, and then returns to the caller's remaining neighbors. The control pattern connects to tree traversal, but a graph can have shared destinations and cycles. A discovered set prevents following those connections forever.

Use A with neighbors B and C, B connected to D, and C also connected to D. With B considered first, a recursive DFS can enter A, B, D and then C if the adjacency order supplies that route. When the call from D has completed, execution resumes in the suspended neighbor loop of its caller. A plain stack of vertices can provide a valid depth-first exploration policy, but reproducing exact recursive entry/exit timing requires storing enough progress to know which neighbor a suspended frame should consider next.

| Event | Meaning |
| --- | --- |
| Enter a new vertex | Mark discovered; begin its neighbor sequence |
| Encounter an already discovered vertex | Avoid a second discovery; interpret the edge under the graph model |
| Return from a child | Resume the parent's remaining neighbors |
| Exit a vertex | Its chosen neighbor sequence is complete |

Do not infer a directed cycle merely from seeing an edge to any previously visited vertex. That destination might already have completed. Directed-cycle reasoning distinguishes an active unfinished path from completed exploration. In an undirected graph, the edge back to the parent also needs special treatment. A single Boolean visited flag solves repeated discovery; additional state or parent information may be needed for the specific cycle question.

For related explanation, see the further reading below.[^development-reading]


```{index} Connectivity Depends on Direction
```



```{index} Connectivity Depends on Direction
```

## Connectivity Depends on Direction



For a symmetric undirected adjacency model, repeated traversals from undiscovered vertices partition the graph into connected components. Every vertex belongs to exactly one such component, and an isolated vertex forms a component by itself. Directed reachability is not symmetric: A may reach B even when B cannot reach A. Repeating ordinary forward searches in arbitrary order does not automatically compute strongly connected components.

Verification can check that each discovered vertex enters/exits once, parent-child call intervals are nested under the recursive model, and every reported reachable vertex has a path from the source. It should not assume that DFS yields minimum-hop paths. BFS has that guarantee under its unweighted model; DFS's first route may be longer.

Guided comparison: draw a graph where A connects directly to D but also begins a long chain that eventually reaches D. Process the chain neighbor first. Predict a DFS route and a BFS route. Then reverse one edge in the chain and identify which reachability claims must be reconsidered. Explain the observed difference using the frontier policy and edge directions rather than the generic phrase graph search.


```{index} Depth-First Search and Cycles
```

## Depth-First Search and Cycles




```{index} Depth-First Search and Cycles
```


Depth-first search (DFS) follows a branch before returning to another neighbor. Mark before following neighbors: a cycle then leads to an already visited vertex rather than infinite recursion. This teaching implementation uses recursive calls; the runtime call stack records unfinished work. It can require O(V) stack space and very deep graphs may exceed runtime stack limits. Part VII develops further traversal techniques. DFS discovers reachability, but its discovery depth does not promise a shortest route.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
Console.WriteLine(string.Join(", ", graph.Dfs("Lobby")));
Console.WriteLine(string.Join(", ", graph.Bfs("Lobby").VisitOrder()));

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby, Hall, Lab, Office
Lobby, Hall, Lab, Office


```{index} Connected Components and Direction
```

## Connected Components and Direction




```{index} Connected Components and Direction
```


For an undirected graph, start a traversal at each still-unvisited vertex. Each new start discovers one connected component. An isolated vertex forms its own component; an empty graph has none. A directed reachability result depends on the starting vertex, so it is not automatically an undirected connected component. The Components method rejects directed graphs rather than silently choosing a different definition.


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
foreach (string[] component in graph.Components()) Console.WriteLine(string.Join(", ", component));
SimpleGraph directed = new SimpleGraph(true);
directed.AddVertex("Approve"); directed.AddVertex("Pay"); directed.AddEdge("Approve", "Pay");
Console.WriteLine(directed.Bfs("Pay").DistanceTo("Approve") is null);

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Lobby, Hall, Lab, Office
Remote
True


```{index} Costs and Contracts
```

## Costs and Contracts




```{index} Costs and Contracts
```


With adjacency lists and expected constant-time dictionary and set operations, a complete traversal takes O(V+E) time: each vertex is discovered once and each stored adjacency is examined once. An undirected edge is stored twice, which changes a constant factor. Visited state, predecessor state, and the queue or recursion stack require O(V) auxiliary space. Matrix traversal instead scans V potential neighbors per visited vertex and can take O(V²). Graph snapshots returned by Vertices and Neighbors protect the stored lists; the strings themselves are immutable. Unknown sources are rejected. The examples do not modify the graph during a search.


```{index} Practice: Check a Reachability Report
```

## Practice: Check a Reachability Report




```{index} Practice: Check a Reachability Report
```


Complete the assertion for the two-edge route to Lab, then add a check for a missing route to Remote.


In [ ]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
if (false /* TODO: detect the wrong Lab distance */ || result.DistanceTo("Remote") is not null) throw new Exception("Reachability failed.");
if (result.PathTo("Lobby").Length != 1) throw new Exception("Source path failed.");
Console.WriteLine("Layer and path checks passed.");

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


In [1]:
using System;
using System.Collections.Generic;

SimpleGraph graph = new SimpleGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Remote" }) graph.AddVertex(room);
graph.AddEdge("Lobby", "Hall"); graph.AddEdge("Hall", "Lab");
graph.AddEdge("Hall", "Office"); graph.AddEdge("Office", "Lab");
SearchResult result = graph.Bfs("Lobby");
if (result.DistanceTo("Lab") != 2 || result.DistanceTo("Remote") is not null) throw new Exception("Reachability failed.");
if (result.PathTo("Lobby").Length != 1) throw new Exception("Source path failed.");
Console.WriteLine("Layer and path checks passed.");

public class SimpleGraph
{
    private readonly Dictionary<string, List<string>> adjacency
        = new Dictionary<string, List<string>>(StringComparer.Ordinal);
    private readonly List<string> vertexOrder = new List<string>();
    public bool IsDirected { get; }
    public SimpleGraph(bool directed = false) { IsDirected = directed; }
    public void AddVertex(string vertex)
    {
        if (string.IsNullOrWhiteSpace(vertex)) throw new ArgumentException("Vertex required.");
        if (adjacency.ContainsKey(vertex)) return;
        adjacency.Add(vertex, new List<string>());
        vertexOrder.Add(vertex);
    }
    public void AddEdge(string from, string to)
    {
        RequireVertex(from); RequireVertex(to);
        if (!adjacency[from].Contains(to)) adjacency[from].Add(to);
        if (!IsDirected && !adjacency[to].Contains(from)) adjacency[to].Add(from);
    }
    public string[] Vertices() { return vertexOrder.ToArray(); }
    public string[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }
    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new ArgumentException("Unknown vertex: " + vertex);
    }
    public SearchResult Bfs(string source)
    {
        RequireVertex(source);
        var distance = new Dictionary<string, int>(StringComparer.Ordinal);
        var previous = new Dictionary<string, string?>(StringComparer.Ordinal);
        var order = new List<string>();
        var waiting = new Queue<string>();
        distance.Add(source, 0); previous.Add(source, null); waiting.Enqueue(source);
        while (waiting.Count > 0)
        {
            string current = waiting.Dequeue();
            order.Add(current);
            foreach (string next in adjacency[current])
            {
                if (distance.ContainsKey(next)) continue;
                distance.Add(next, distance[current] + 1);
                previous.Add(next, current);
                waiting.Enqueue(next);
            }
        }
        return new SearchResult(distance, previous, order);
    }
    public string[] Dfs(string source)
    {
        RequireVertex(source);
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var order = new List<string>();
        Visit(source, seen, order);
        return order.ToArray();
    }
    private void Visit(string current, HashSet<string> seen, List<string> order)
    {
        if (!seen.Add(current)) return;
        order.Add(current);
        foreach (string next in adjacency[current]) Visit(next, seen, order);
    }
    public List<string[]> Components()
    {
        if (IsDirected) throw new InvalidOperationException("Components requires an undirected graph.");
        var seen = new HashSet<string>(StringComparer.Ordinal);
        var groups = new List<string[]>();
        foreach (string vertex in vertexOrder)
        {
            if (seen.Contains(vertex)) continue;
            string[] group = Bfs(vertex).VisitOrder();
            foreach (string member in group) seen.Add(member);
            groups.Add(group);
        }
        return groups;
    }
}

public class SearchResult
{
    private readonly Dictionary<string, int> distance;
    private readonly Dictionary<string, string?> previous;
    private readonly List<string> order;
    internal SearchResult(Dictionary<string, int> distance,
        Dictionary<string, string?> previous, List<string> order)
    {
        this.distance = distance; this.previous = previous; this.order = order;
    }
    public int? DistanceTo(string target)
    {
        return distance.TryGetValue(target, out int value) ? value : null;
    }
    public string[] VisitOrder() { return order.ToArray(); }
    public string[] PathTo(string target)
    {
        if (!distance.ContainsKey(target)) return Array.Empty<string>();
        var reverse = new List<string>();
        string? current = target;
        while (current is not null)
        {
            reverse.Add(current);
            current = previous[current];
        }
        reverse.Reverse();
        return reverse.ToArray();
    }
}


Layer and path checks passed.


(traversal-verification)=

```{index} Verify a Traversal
```

## Verify a Traversal


The breadth-first and depth-first searches developed in this chapter also provide evidence that can be checked independently. Use these closing traversal-verification contracts before applying them to later network-coverage and algorithm-design tasks.


```{index} Traversal Verification Contracts
```

## Traversal Verification Contracts




```{index} graph traversal; verification
```


Verify a BFS result by checking that the source has distance zero, each reachable non-source has a predecessor one layer earlier, and every examined edge satisfies the layer bound. Check unreachable vertices separately. A shortest path in edge count need not minimize weighted cost; Chapter 24 supplies Dijkstra’s algorithm for nonnegative weights.

For DFS, check that each reached vertex enters and exits exactly once, every child interval is nested in its parent interval, and cycles do not cause repeated discovery. An explicit stack of suspended neighbor positions reproduces recursive order; a stack that pushes all neighbors at once may produce another order.


```{index} Components and Direction
```

## Components and Direction




```{index} connected components; directed graph
```


The component method in Chapter 20 requires symmetric adjacency. A directed reachability search does not compute strongly connected components. Choose the mathematical contract before selecting an implementation. Use the Chapter 26 lab to connect traversal evidence with network coverage and the limits of heuristic routing.


```{rubric} Footnotes
```


[^2601_graph_traversal-1]: BFS minimizes edge count. Equal positive edge costs preserve the same ordering of path costs; arbitrary weights require a different algorithm.
[^2601_graph_traversal-2]: A directed graph needs a deliberately chosen notion such as weak or strong connectivity; neither is computed by this undirected component method.

[^2003_graph_traversal-1]: BFS minimizes the number of edges only when edges have equal cost. Weighted shortest paths are developed in Chapter 24.


[^development-reading]: Further reading: [OpenDSA, Graph Traversals](https://opendsa.org/OpenDSA/Books/Catalog/html/GraphTraversal.html). The worked business scenarios and guided reasoning here are original teaching examples.
